# 📨 9장 · 스팸일까, 아닐까: 분류 (개정판 실습 노트북)

『빅데이터분석 with 파이썬』 9장의 **05. 파이썬 실습**과 **미니 프로젝트**를 그대로 따라 하는 노트북입니다. 실습 번호는 책과 같습니다.

| 표시 | 뜻 | 실습 |
|---|---|---|
| ★ 필수 | 수업 시간에 모두 함께 | 9-1, 9-3, 9-4, 9-5, 9-6, 9-7 |
| ☆ 선택 | 자습이나 심화 | 9-2, 9-8, 9-9 |

**데이터**
- 스팸 문자: UCI "SMS Spam Collection" 5,574통(중복을 지우면 5,171통). GitHub 저장소 `justmarkham/pycon-2016-tutorial`에서 바로 읽습니다.
- 미니 프로젝트: 서울 초미세먼지(한국환경공단 에어코리아 확정자료, 도시대기 측정소 25곳의 시각별 평균)와 날씨(기상청 ASOS 서울 108 시간자료), 2023년 1월~2026년 4월 29,184시각. 출처 한국환경공단·기상청, 공공누리. 두 파일 `에어코리아_서울_도시대기평균_시간별_2023_2026.csv`, `asos_서울108_시간자료_2023_2026.csv`는 강의 자료에 들어 있습니다(13장 종합 프로젝트와 같은 파일). 코랩 왼쪽 파일 창에 올려 두거나, 미니 프로젝트 1단계 앞의 셀로 올립니다. 에어코리아 달별 원자료를 직접 줄이는 ☆ 셀도 있습니다(받은 파일이 없으면 건너뜁니다).

코드 셀은 **위에서부터 차례로** 실행하십시오(`Shift + Enter`). 아래 셀은 위에서 만든 변수를 이어서 씁니다.

## 0 · 준비

그래프의 한글이 네모(□)로 깨지지 않게 해 주는 **koreanize-matplotlib**를 설치합니다. 판다스, 사이킷런, 맷플롯립은 코랩에 이미 들어 있습니다.

In [ ]:
!pip install -q koreanize-matplotlib

**데이터 받기** (깃허브 저장소판에서 덧붙인 셀) — 아래 셀은 이 장이 쓰는 강의 자료 파일을 저장소의 `data` 폴더에서 코랩으로 내려받습니다.

In [ ]:
# 이 장의 강의 자료 파일을 책 깃허브 저장소에서 받아 옵니다.
# (파일을 코랩 왼쪽 📁 파일 창에 직접 끌어다 올렸다면 이 셀은 건너뛰어도 됩니다.)
import os, urllib.parse, urllib.request
BASE = "https://raw.githubusercontent.com/yeonu69/bigdata-python-book/main/ch09/data/"
for f in [
    "asos_서울108_시간자료_2023_2026.csv",
    "에어코리아_서울_도시대기평균_시간별_2023_2026.csv",
]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(BASE + urllib.parse.quote(f), f)
    print("준비됨:", f, f"({os.path.getsize(f) / 1e6:.1f} MB)")

## 05. 파이썬 실습: 스팸 문자 가려내기

이 절의 실습은 코랩(Colab)에서 위에서부터 차례로 실행합니다. 문자 데이터는 인터넷 주소에서 바로 읽습니다. 분류 모델은 사이킷런으로 만듭니다. 사이킷런의 모델은 모두 `fit( )`으로 배우고 `predict( )`로 판정합니다. 분류 모델은 `predict_proba( )`로 확률도 냅니다. ★는 수업 시간에 꼭 돌리는 실습이고, ☆는 자습용입니다.

#### [파이썬 실습 9-1 ★] 문자 데이터 불러오기와 기준선

문자 데이터를 읽어 똑같은 문자를 지우고, 스팸 비율과 기준선 정확도를 구하시오. 숫자 개수 열도 만들어 두시오.

① 한글 글꼴을 설정하고 데이터를 읽습니다. ② 중복을 지우고 스팸 비율을 셉니다. ③ "스팸" 열(1, 0)과 "숫자" 열(숫자 개수)을 만듭니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib            # 코랩: 한글 글꼴을 자동으로 설정
except ImportError:                        # 나눔고딕 파일이 이미 있는 컴퓨터
    import glob, os
    import matplotlib.font_manager as fm
    for f in glob.glob(os.path.expanduser("~/.fonts/Nanum*.ttf")):
        fm.fontManager.addfont(f)
    plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

URL = ("https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/"
       "master/data/sms.tsv")
sms = pd.read_csv(URL, sep="\t", header=None, names=["label", "msg"], quoting=3)
print("처음 읽은 문자:", len(sms), "통,  똑같은 문자:", sms.duplicated().sum(), "통")
sms = sms.drop_duplicates().reset_index(drop=True)
print(sms["label"].value_counts().to_string())
print("스팸 비율:", round((sms["label"] == "spam").mean(), 3))
print("모두 정상이라고 찍을 때 정확도:", round((sms["label"] == "ham").mean(), 3))

sms["스팸"] = (sms["label"] == "spam").astype(int)  # 스팸이면 1, 정상이면 0
sms["숫자"] = sms["msg"].str.count(r"\d")          # 문자 속 숫자(0~9)의 개수
print(sms.groupby("label")["숫자"].mean().round(1).to_string())
print("평균 글자 수:", sms["msg"].str.len().groupby(sms["label"]).mean().round(1).to_dict())

sms["label"].value_counts().plot.bar(color=["#8EC5E8", "#E04B3C"], figsize=(4, 3))
plt.title("정상 문자와 스팸 문자의 수")
plt.xticks(rotation=0)
plt.show()

5,574통 가운데 똑같은 문자가 403통이었습니다. 지우고 나니 5,171통, 스팸은 12.6%입니다. 그래서 "모두 정상"이라고 찍기만 해도 87.4%를 맞힙니다. 앞으로 만드는 모델은 이 숫자를 넘어야 뜻이 있습니다.

> **[코드 읽기] `quoting=3`과 `str.count(r"\d")`**
>
> `quoting=3`은 "문자 속 따옴표를 그냥 글자로 읽어라"는 뜻입니다. 이것을 빼면 따옴표가 든 문자 여러 통이 한 줄로 붙어 행 수가 달라집니다. `str.count(r"\d")`는 문자마다 숫자(0~9)가 몇 번 나오는지 셉니다. `\d`는 "숫자 한 글자"를 뜻하는 정규식입니다.

#### [파이썬 실습 9-2 ☆] 숫자 개수로 나누어 보기

숫자 개수를 구간으로 나누어, 구간마다 문자 수와 스팸 비율을 구하시오.

In [ ]:
bins = [-1, 0, 2, 5, 10, 20, 100]
group = pd.cut(sms["숫자"], bins)
table = sms.groupby(group, observed=True)["스팸"].agg(["size", "mean"])
print(table.round(3).to_string())
rate = table["mean"]

rate.plot.bar(color="#E04B3C", figsize=(6, 3))
plt.ylabel("스팸 비율")
plt.xlabel("문자 속 숫자 개수")
plt.xticks(rotation=0)
plt.show()

`pd.cut( )`은 숫자를 구간으로 나눕니다. `(2, 5]`는 "2 초과 5 이하", 곧 3~5개입니다. "스팸" 열이 0과 1이므로, 평균(mean)이 곧 스팸 비율입니다. 숫자가 없는 문자 3,842통은 스팸이 1.0%뿐이고, 11개 이상이면 거의 모두 스팸입니다.

#### [파이썬 실습 9-3 ★] 숫자 개수 하나로 로지스틱 회귀

문자를 훈련용과 시험용으로 나누고, 숫자 개수 하나로 로지스틱 회귀를 맞추시오. 오즈비와 확률이 0.5가 되는 숫자 개수를 구하고, 시그모이드 곡선을 그리시오.

① 75%와 25%로 나눕니다. ② 로지스틱 회귀를 맞추고 절편과 기울기를 꺼냅니다. ③ (식 9-1)로 확률을 직접 계산해 그립니다.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

train, test = train_test_split(sms, test_size=0.25, random_state=42,
                               stratify=sms["스팸"])          # 75%로 배우고 25%로 시험
print("훈련", len(train), "통 / 시험", len(test), "통")

one = LogisticRegression().fit(train[["숫자"]], train["스팸"])
b0, b1 = one.intercept_[0], one.coef_[0][0]
print(f"절편 {b0:.2f}, 기울기 {b1:.3f}")
print("숫자 하나가 늘 때 오즈는", round(np.exp(b1), 2), "배")
print("확률이 0.5가 되는 숫자 개수:", round(-b0 / b1, 1))

x = np.arange(0, 21)
p = 1 / (1 + np.exp(-(b0 + b1 * x)))          # 시그모이드
for k in [0, 2, 5, 10, 15]:
    print(f"숫자 {k:2d}개 → 스팸일 확률 {p[k]:.3f}")

plt.figure(figsize=(6, 3))
plt.plot(x, p, color="#E04B3C")
plt.axhline(0.5, color="gray", linestyle="--")
plt.xlabel("문자 속 숫자 개수")
plt.ylabel("스팸일 확률")
plt.show()

`stratify`는 훈련과 시험의 스팸 비율을 똑같이 맞추어 줍니다. 기울기 0.643에 e를 씌운 1.9가 오즈비입니다. 확률이 0.5가 되는 곳은 z = 0인 곳, 곧 −b₀ ÷ b₁ = 6.3개입니다. 숫자가 15개면 확률이 0.996으로, 거의 확실히 스팸이라고 봅니다.

> **[코드 읽기] f-문자열의 `:.3f`**
>
> `f"절편 {b0:.2f}"`는 3장에서 본 f-string입니다. `:.2f`는 "소수점 아래 둘째 자리까지", `:2d`는 "정수를 두 칸에 맞추어"라는 뜻입니다.

#### [파이썬 실습 9-4 ★] 혼동행렬과 정밀도, 재현율

숫자 개수 모델로 시험 문자를 판정하고, 혼동행렬, 정확도, 정밀도, 재현율, F1을 구하시오.

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score

pred = one.predict(test[["숫자"]])            # 문턱값 0.5로 판정
y = test["스팸"]
print(confusion_matrix(y, pred))             # [[TN FP] [FN TP]]
print("정확도:", round(accuracy_score(y, pred), 3))
print("정밀도:", round(precision_score(y, pred), 3))
print("재현율:", round(recall_score(y, pred), 3))
print("F1    :", round(f1_score(y, pred), 3))

from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_predictions(y, pred, display_labels=["정상", "스팸"],
                                        cmap="Blues", colorbar=False)
plt.xlabel("모델의 판정")
plt.ylabel("실제")
plt.show()

사이킷런의 혼동행렬은 첫 줄이 실제 정상, 둘째 줄이 실제 스팸입니다. 왼쪽 칸이 "정상" 판정, 오른쪽 칸이 "스팸" 판정입니다. 헛경보는 2통뿐이라 정밀도가 98.5%로 높습니다. 하지만 스팸 31통을 놓쳐 재현율은 81.0%입니다. 숫자가 적은 스팸이 빠져나간 것입니다.

#### [파이썬 실습 9-5 ★] 단어로 판정하기: 나이브 베이즈와 로지스틱 회귀

문자를 단어 횟수로 바꾸고, 나이브 베이즈와 로지스틱 회귀의 성적을 숫자 개수 모델과 견주시오.

① `CountVectorizer`로 단어마다 횟수를 셉니다. ② 두 모델을 맞춥니다. ③ 세 모델의 성적을 표와 그래프로 견줍니다.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

low = sms["msg"].str.lower()                    # 단어가 든 비율은 문자 전체에서 센다
for w in ["free", "call"]:
    has = low.str.contains(rf"\b{w}\b")
    print(f'"{w}" 단어가 든 비율: 스팸 {has[sms["스팸"] == 1].mean():.1%}, 정상 {has[sms["스팸"] == 0].mean():.1%}')
both = low.str.contains(r"\bfree\b") & low.str.contains(r"\bcall\b")
print("두 단어가 모두 든 문자:", both.sum(), "통, 그 가운데 스팸", f'{sms.loc[both, "스팸"].mean():.1%}')

vec = CountVectorizer()                       # 단어마다 몇 번 나왔는지 센다
A_tr = vec.fit_transform(train["msg"])
A_te = vec.transform(test["msg"])
print("단어 종류:", A_tr.shape[1])
i = vec.vocabulary_["prize"]
is_spam = (train["스팸"] == 1).values
print('훈련 문자의 "prize": 스팸에', A_tr[is_spam][:, i].sum(), "번, 정상에", A_tr[~is_spam][:, i].sum(), "번")

nb = MultinomialNB().fit(A_tr, train["스팸"])
lr = LogisticRegression(max_iter=2000).fit(A_tr, train["스팸"])

rows = []
for name, m, X in [("숫자 개수 하나", one, test[["숫자"]]),
                   ("나이브 베이즈(단어)", nb, A_te),
                   ("로지스틱 회귀(단어)", lr, A_te)]:
    pr = m.predict(X)
    rows.append([name, accuracy_score(y, pr), precision_score(y, pr),
                 recall_score(y, pr), f1_score(y, pr)])
result = pd.DataFrame(rows, columns=["모델", "정확도", "정밀도", "재현율", "F1"])
print(result.round(3).to_string(index=False))

result.set_index("모델")[["정확도", "재현율"]].plot.barh(figsize=(6, 3),
                                                    color=["#8EC5E8", "#E04B3C"])
plt.axvline(0.874, color="gray", linestyle="--")   # 모두 정상이라고 찍을 때
plt.xlim(0, 1)
plt.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
plt.show()

`CountVectorizer`는 훈련 문자에 나온 단어 7,395종류로 사전을 만들고, 문자마다 단어별 횟수를 셉니다. 시험 문자는 `transform( )`으로 같은 사전에 맞추어 바꿉니다. 훈련 때 없던 단어는 무시됩니다. 먼저 찍힌 세 줄은 1.4와 4.4에서 쓴 숫자입니다. "free"는 스팸의 22.7%, 정상 문자의 1.2%에 들어 있고, 두 단어가 모두 든 64통 가운데 스팸은 81.2%입니다. "prize"는 훈련 스팸에 62번 나왔지만 정상 문자에는 없습니다. 정확도는 세 모델이 비슷하지만, 재현율은 나이브 베이즈가 91.4%로 가장 높습니다. 그래프로 보면 파란 막대는 모두 비슷하고, 빨간 막대에서 차이가 납니다.

#### [파이썬 실습 9-6 ★] 문턱값 바꾸기

로지스틱 회귀의 확률을 받아 문턱값을 0.1에서 0.9까지 바꾸며 헛경보, 놓침, 손해(헛경보 × 10 + 놓침)를 구하시오.

In [ ]:
prob = lr.predict_proba(A_te)[:, 1]           # 스팸일 확률
rows = []
for th in [0.1, 0.3, 0.5, 0.7, 0.9]:
    tn, fp, fn, tp = confusion_matrix(y, (prob >= th).astype(int)).ravel()
    rows.append([th, fp, fn, 10 * fp + fn])
cost = pd.DataFrame(rows, columns=["문턱값", "헛경보", "놓침", "손해(헛경보×10+놓침)"])
print(cost.to_string(index=False))

cost.plot(x="문턱값", y=["헛경보", "놓침"], marker="o", figsize=(6, 3),
          color=["#3F7CC0", "#E04B3C"])
plt.ylabel("문자 수")
plt.show()

`predict( )`는 늘 문턱값 0.5로 판정합니다. 다른 문턱값을 쓰려면 `predict_proba( )`로 확률을 받아 직접 견줍니다. `[:, 1]`은 두 열(정상일 확률, 스팸일 확률) 가운데 둘째 열을 고르는 것입니다. 헛경보를 열 배로 치면 0.9의 손해가 50으로 가장 작습니다.

#### [파이썬 실습 9-7 ★] 모델이 배운 단어 들여다보기

로지스틱 회귀의 계수가 가장 큰 단어와 가장 작은 단어를 찾고, "lor", "da", `&lt;#&gt;`가 스팸과 정상 문자에 몇 번 나오는지 세시오.

In [ ]:
words = vec.get_feature_names_out()
coef = lr.coef_[0]
order = np.argsort(coef)
print("정상 쪽 단어:", list(words[order[:8]]))
print("스팸 쪽 단어:", list(words[order[-8:]][::-1]))

low = sms["msg"].str.lower()
for w in ["lor", "da", "&lt;#&gt;"]:
    has = low.str.contains(w, regex=False) if w.startswith("&") else low.str.contains(rf"\b{w}\b")
    print(f"'{w}' 든 문자: 스팸 {has[sms['스팸'] == 1].sum()}통, 정상 {has[sms['스팸'] == 0].sum()}통")

top = pd.Series(coef[order[[0, 1, 2, 3, 4, -5, -4, -3, -2, -1]]],
                index=words[order[[0, 1, 2, 3, 4, -5, -4, -3, -2, -1]]])
top.plot.barh(color=["#3F7CC0"] * 5 + ["#E04B3C"] * 5, figsize=(5, 4))
plt.xlabel("계수 (오른쪽일수록 스팸 쪽)")
plt.show()

`np.argsort( )`는 값을 작은 것부터 늘어놓았을 때의 순서를 돌려줍니다. 그래서 앞쪽은 정상 쪽, 뒤쪽은 스팸 쪽 단어입니다. 정상 쪽에 "gt"와 "lt"가 있습니다. 그리고 세 표시는 모두 정상 문자에만 나옵니다. 모델이 데이터의 출신을 배웠다는 3.6의 이야기를 코드로 확인한 것입니다.

#### [파이썬 실습 9-8 ☆] 데이터가 적을 때는 누가 이기나

훈련 문자를 50통, 200통, 1,000통으로 줄여 열 번씩 다시 배우고, 두 모델의 평균 F1을 견주시오.

In [ ]:
rows = []
for n in [50, 200, 1000]:
    f_nb, f_lr = [], []
    for seed in range(10):                    # 무작위로 n통을 10번 뽑아 평균
        sub = train.sample(n, random_state=seed)
        v = CountVectorizer().fit(sub["msg"])
        a, b = v.transform(sub["msg"]), v.transform(test["msg"])
        f_nb.append(f1_score(y, MultinomialNB().fit(a, sub["스팸"]).predict(b)))
        f_lr.append(f1_score(y, LogisticRegression(max_iter=2000).fit(a, sub["스팸"]).predict(b)))
    rows.append([n, np.mean(f_nb), np.mean(f_lr)])
small = pd.DataFrame(rows, columns=["훈련 문자 수", "나이브 베이즈 F1", "로지스틱 F1"])
print(small.round(3).to_string(index=False))

훈련 문자가 50통이면 스팸은 6통 남짓뿐입니다. 로지스틱 회귀는 그 적은 예로 수백 개의 계수를 정하지 못해, 거의 모든 문자를 정상으로 판정했습니다(F1 0.127). 나이브 베이즈는 단어 횟수만 세면 되므로 0.601까지 갔습니다. 데이터가 늘수록 차이는 좁혀집니다. 쉽게 말하면, **데이터가 적을 때는 가정이 강한 단순한 모델이 버티고, 데이터가 쌓이면 다른 모델이 따라잡습니다.**

#### [파이썬 실습 9-9 ☆] ROC 곡선과 AUC

숫자 개수 모델과 단어 로지스틱 회귀의 ROC 곡선을 그리고 AUC를 견주시오.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

for name, pr in [("숫자 개수 하나", one.predict_proba(test[["숫자"]])[:, 1]),
                 ("로지스틱 회귀(단어)", prob)]:
    fpr, tpr, _ = roc_curve(y, pr)
    plt.plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(y, pr):.3f})")
    print(name, "AUC:", round(roc_auc_score(y, pr), 3))
plt.plot([0, 1], [0, 1], color="gray", linestyle="--")
plt.xlabel("헛경보 비율")
plt.ylabel("재현율")
plt.legend()
plt.show()

두 곡선 모두 왼쪽 위 모서리에 바짝 붙어 있습니다. AUC는 숫자 개수 모델이 0.965, 단어 모델이 0.982입니다. 단어 모델이 스팸과 정상을 조금 더 잘 가릅니다.

> **핵심 3줄**
>
> 사이킷런에서는 데이터를 나누고, 모델을 배우고(`fit`), 확률을 내고(`predict_proba`), 혼동행렬과 정밀도·재현율로 채점하는 순서가 늘 같습니다.
>
> 훈련 데이터가 적으면 나이브 베이즈가 버티고, 문턱값과 상관없이 두 모델을 견줄 때는 ROC 곡선과 AUC를 봅니다.
>
> 훈련 문자 50통에서 F1은 나이브 베이즈 0.601, 로지스틱 회귀 0.127이었고, AUC는 단어 모델 0.982, 숫자 개수 모델 0.965였습니다.

## [미니 프로젝트 9] 날씨만 보고 서울 초미세먼지 "나쁨"을 맞힐 수 있을까

**질문.** 천식이 있는 친구에게 "지금은 마스크를 꼭 쓰세요" 알림을 보내는 앱을 만든다고 합시다. 서울의 날씨만 보고 초미세먼지(PM2.5)가 "나쁨"인 시각을 맞힐 수 있을까요? 날씨는 내일 것까지 예보가 나오지만, 미세먼지 측정값은 지금 것까지만 있습니다. 그래서 먼저 날씨가 같은 시각의 미세먼지를 얼마나 알려 주는지부터 확인합니다. 오늘의 값으로 내일을 맞히는 더 어려운 질문은 13장 종합 프로젝트의 주제 C에서 이어 갈 것입니다.

**데이터.** 두 파일을 씁니다. 하나는 한국환경공단 에어코리아의 확정자료입니다. 서울의 도시대기 측정소 25곳이 잰 PM2.5를 시각마다 평균했습니다. 2023년 1월 1일 01시부터 2026년 4월 30일 24시까지 29,184행이고, 한 행은 "한 시각"입니다. 다른 하나는 기상청 서울 관측소(지점 번호 108)의 시간자료로, 기온, 강수량, 풍속, 습도가 들어 있습니다. 두 파일은 13장 종합 프로젝트와 같은 파일로, 강의 자료에 들어 있습니다. Colab 왼쪽 파일 창에 끌어다 놓고 시작합니다. "나쁨"은 환경부의 예보 등급(36~75㎍/㎥)을 빌려 우리가 정했습니다. 평균이 35㎍/㎥를 넘는 시각을 모두 "나쁨"으로 셉니다. 서울에서 "매우 나쁨"(76 이상)은 너무 드물기 때문입니다.

> **[더 알아보기] ☆ 에어코리아에서 직접 받아 줄이기**
>
> 에어코리아 누리집의 확정자료는 달마다 엑셀 파일이 하나씩입니다. 파일 하나에 전국 측정소의 한 달 기록이 약 48만 행 들어 있습니다. 강의 자료의 파일은 그 가운데 서울의 도시대기 측정소만 골라 시각마다 평균한 것입니다. 도로변 측정소 15곳은 자동차 바로 옆의 공기를 재므로 뺐습니다. 직접 줄일 때 챙길 것이 하나 있습니다. 에어코리아는 하루를 01시~24시로 적습니다. 24시는 다음 날 00시이므로, 00시~23시로 적는 날씨 파일과 맞추려면 날짜를 하루 넘겨야 합니다. 날씨는 기상자료개방포털의 "종관기상관측(ASOS) 시간자료"에서 서울(108)을 골라 받습니다. 여러 번 나누어 받았다면 위아래로 잇고(`pd.concat`), 같은 시각은 하나만 남깁니다(`drop_duplicates`).

In [ ]:
# ☆ 에어코리아에서 받은 달별 원자료를 서울 도시대기 평균 한 파일로 줄이기 (강의 자료의 파일을 쓰면 건너뜁니다)
import glob
RAW = sorted(glob.glob("20*년 *월.xlsx"))          # 받은 파일 이름에 맞게 고칩니다
if not RAW:
    print("에어코리아에서 받은 파일이 없습니다. 강의 자료의 파일로 1단계를 하십시오.")
else:
    parts = []
    for f in RAW:                                   # 파일 하나가 전국 약 48만 행이라 1분쯤 걸린다
        t = pd.read_excel(f, usecols=["지역", "망", "측정일시", "PM25"])
        parts.append(t[t["지역"].str.startswith("서울") & (t["망"] == "도시대기")])
    raw = pd.concat(parts)
    h = raw.groupby("측정일시")["PM25"].agg(PM25="mean", 측정소수="count").reset_index()
    date = pd.to_datetime(h["측정일시"].astype(str).str[:8])          # 2023010124 → 2023-01-01
    h["일시"] = date + pd.to_timedelta(h["측정일시"] % 100, unit="h")  # + 24시간 → 다음 날 0시
    h["PM25"] = h["PM25"].round(2)
    h = h[["측정일시", "일시", "PM25", "측정소수"]]
    h.to_csv("에어코리아_직접_줄인_파일.csv", index=False)   # 1단계의 AIR를 이 이름으로
    print(len(RAW), "개 파일 →", h.shape, h["일시"].min(), "~", h["일시"].max())

**1단계: 데이터 신원 확인**([파이썬 실습 9-1] 응용). 두 파일을 읽고 기간, 빈칸, 시각 표기를 확인하시오. 같은 시각끼리 합치고 "나쁨" 열을 만든 뒤, 달마다 "나쁨" 비율을 구하시오.

In [ ]:
# 파일 창에 미니 프로젝트의 두 CSV를 올리지 않았다면, 이 셀이 올릴 창을 엽니다.
import os
if not all(os.path.exists(f) for f in ['에어코리아_서울_도시대기평균_시간별_2023_2026.csv', 'asos_서울108_시간자료_2023_2026.csv']):
    from google.colab import files
    files.upload()

In [ ]:
AIR = "에어코리아_서울_도시대기평균_시간별_2023_2026.csv"   # Colab 왼쪽 파일 창에 올린 두 파일
WEA = "asos_서울108_시간자료_2023_2026.csv"
air = pd.read_csv(AIR, parse_dates=["일시"])
wea = pd.read_csv(WEA, parse_dates=["일시"])
print("초미세먼지:", air.shape, air["일시"].min(), "~", air["일시"].max())
print("측정소 수:", air["측정소수"].min(), "~", air["측정소수"].max(), "곳")
print(air[["측정일시", "일시"]].iloc[22:24].to_string(index=False))  # 24시 = 다음 날 0시
print("날씨:", wea.shape, " 빈칸:", wea.isna().sum().to_dict())

wea.columns = ["지점", "지점명", "일시", "기온", "강수량", "풍속", "습도"]
wea["강수량"] = wea["강수량"].fillna(0)              # 비가 오지 않은 시각은 빈칸이다
air = air.merge(wea[["일시", "기온", "강수량", "풍속", "습도"]], on="일시")  # 같은 시각끼리
air["나쁨"] = (air["PM25"] > 35).astype(int)         # 예보 등급의 "나쁨"은 36㎍/㎥부터
print("합친 표:", air.shape, " 나쁨 비율:", round(air["나쁨"].mean(), 3))
print("매우 나쁨(76 이상) 비율:", round((air["PM25"] > 75).mean(), 3))
month = air.groupby(air["일시"].dt.month)["나쁨"].mean()
print(month.round(3).to_dict())

month.plot.bar(color="#E04B3C", figsize=(6, 3))
plt.xlabel("달")
plt.ylabel("나쁨 비율")
plt.xticks(rotation=0)
plt.show()

먼저 시각 표기를 봅니다. 측정일시 2023010124는 일시 열에서 2023-01-02 00:00입니다. 에어코리아의 24시를 다음 날 0시로 바꾸어 둔 것입니다. 그래서 날씨 파일과 일시 열로 바로 맞출 수 있습니다. 측정소 수가 17~25곳인 것은 값이 빈 측정소를 빼고 평균했기 때문입니다.

빈칸도 뜻이 다릅니다(3장). 강수량의 빈칸 29,375개는 고장이 아니라 "비가 오지 않은 시각"이라 0으로 채웠습니다. 풍속의 빈칸 68개는 진짜로 빠진 기록입니다. 합친 29,184시각 가운데 "나쁨"은 11.4%입니다. 그런데 달마다 크게 다릅니다. 3월은 30.9%, 8~10월은 1%에도 못 미칩니다.

**2단계: 기준선과 모델**([파이썬 실습 9-3], [9-5] 응용). 지난 24시간의 날씨로 변수를 만드시오. 2023년으로 배우고, 2024년으로 문턱값을 고르고, 2025년 1월~2026년 4월로 시험하도록 나누시오. 기준선과 로지스틱 회귀의 정확도를 견주시오.

In [ ]:
air["1시간전"] = air["PM25"].shift(1)              # 바로 앞 시각의 농도 (4단계에서 씀)
for c in ["기온", "습도", "풍속"]:                  # 그 시각까지 24시간의 평균
    air[c + "24"] = air[c].rolling(24).mean()
air["비24"] = (air["강수량"].rolling(24).sum() > 0).astype(int)  # 24시간 안에 비가 왔나
m = air["일시"].dt.month
air["겨울"] = m.isin([12, 1, 2]).astype(int)
air["봄"] = m.isin([3, 4, 5]).astype(int)           # 여름과 가을이 기준
X_COLS = ["기온24", "습도24", "풍속24", "비24", "겨울", "봄"]
air = air.dropna(subset=X_COLS)                     # 처음 23시각, 바람 기록이 빠진 때

yr = air["일시"].dt.year
past = air[yr == 2023]                              # 2023년으로 배우고
valid = air[yr == 2024]                             # 2024년으로 문턱값을 고르고
future = air[yr >= 2025]                            # 2025년 1월~2026년 4월로 한 번 시험
print("훈련", len(past), "/ 검증", len(valid), "/ 시험", len(future), "시각")
yt = future["나쁨"]
print("시험 기간 나쁨 비율:", round(yt.mean(), 3), " 기준선(늘 괜찮음):", round(1 - yt.mean(), 3))

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
air_m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
air_m.fit(past[X_COLS], past["나쁨"])
pt = air_m.predict_proba(future[X_COLS])[:, 1]
print("로지스틱 회귀(문턱값 0.5) 정확도:", round(accuracy_score(yt, pt >= 0.5), 3),
      " 재현율:", round(recall_score(yt, pt >= 0.5), 3))

미세먼지는 한 시각에 생기지 않습니다. 하루에 걸쳐 쌓이고 씻깁니다. 그래서 그 시각의 날씨 대신 그 시각까지 24시간의 평균을 썼습니다. `rolling(24)`는 바로 앞 24행을 묶습니다. 두 파일 모두 한 시각도 빠짐없이 이어져 있어서 24행이 곧 24시간입니다.

데이터를 세 토막으로 나눈 까닭이 있습니다. 내일을 맞힐 모델은 내일로 시험해야 합니다(8장). 그리고 문턱값을 시험 기간으로 고르면, 시험 문제를 보고 답을 고친 셈이 됩니다. 그래서 문턱값은 그 앞의 검증 기간으로 고릅니다.

시험 기간의 "나쁨"은 13.6%이므로, 늘 "괜찮음"이라고 찍어도 86.4%를 맞힙니다. 로지스틱 회귀는 87.4%로, 1%p 남짓 높을 뿐입니다. 재현율은 14.5%입니다. "나쁨"인 시각 일곱 번 가운데 한 번만 알려 준다는 뜻입니다. 정확도로는 기준선을 겨우 넘었습니다.

**3단계: 문턱값 고르기**([파이썬 실습 9-6] 응용). 천식 알림에서는 놓침이 헛경보보다 세 배 나쁘다고 합시다. 검증 기간(2024년)에서 손해가 가장 작은 문턱값을 고르시오. 그 문턱값으로 시험 기간을 한 번만 판정해, 알림을 보내지 않을 때, 늘 보낼 때와 견주시오.

In [ ]:
pv = air_m.predict_proba(valid[X_COLS])[:, 1]       # 검증 기간(2024년)의 확률
yv = valid["나쁨"]
rows = []
for th in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
    tn, fp, fn, tp = confusion_matrix(yv, (pv >= th).astype(int)).ravel()
    rows.append([th, fp, fn, 3 * fn + fp])
air_cost = pd.DataFrame(rows, columns=["문턱값", "헛경보", "놓침", "손해(놓침×3+헛경보)"])
print(air_cost.to_string(index=False))
print("알림을 한 번도 보내지 않으면 손해:", 3 * yv.sum())
best = air_cost.loc[air_cost["손해(놓침×3+헛경보)"].idxmin(), "문턱값"]
print("검증으로 고른 문턱값:", best)

air_cost.plot(x="문턱값", y=["헛경보", "놓침"], marker="o", figsize=(6, 3),
              color=["#3F7CC0", "#E04B3C"])
plt.ylabel("시각 수 (2024년)")
plt.show()

In [ ]:
plans = {"알림 안 보냄": np.zeros(len(yt), int), "늘 알림": np.ones(len(yt), int),
         "문턱값 0.5": (pt >= 0.5).astype(int), f"문턱값 {best}": (pt >= best).astype(int)}
for name, alert in plans.items():                   # 시험 기간에는 딱 한 번 판정한다
    tn, fp, fn, tp = confusion_matrix(yt, alert, labels=[0, 1]).ravel()
    print(f"{name:7s} 정확도 {(tp + tn) / len(yt):.3f}  재현율 {tp / (tp + fn):.3f}  "
          f"놓침 {fn:4d}  헛경보 {fp:5d}  손해 {3 * fn + fp}")

검증 기간에서 손해는 문턱값 0.3일 때 1,963으로 가장 작았습니다. 알림을 한 번도 보내지 않을 때(2,175)보다 작습니다. 이 0.3을 그대로 들고 시험 기간에 갔습니다. 손해는 3,670으로, 알림을 보내지 않을 때(4,746)와 문턱값 0.5(4,176)보다 작습니다. "나쁨"인 시각의 38.3%를 미리 알려 줍니다.

눈여겨볼 것이 있습니다. 문턱값 0.3의 정확도는 85.2%로, 늘 "괜찮음"이라고 찍기(86.4%)보다 낮습니다. 그런데도 손해는 가장 작습니다. 스팸 필터와 반대로, 여기서는 놓침이 비싸므로 문턱값을 내리는 쪽이 낫습니다. 확률이 잘 맞는다면, 놓침의 값 3 × p가 헛경보의 값 1 − p보다 클 때, 곧 p가 0.25를 넘을 때 알리는 것이 이득입니다. 검증으로 고른 0.3은 여기서 멀지 않습니다.

**4단계: 너무 좋은 점수 의심하기.** 1시간 전 PM2.5를 변수로 더하면 정확도가 어떻게 되는지 구하시오. "1시간 전과 같다"고 찍기와 견주고, 이 모델을 알림 앱에 써도 되는지 쓰시오.

In [ ]:
cols = X_COLS + ["1시간전"]
lag = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
lag.fit(past[cols], past["나쁨"])
print("1시간 전 농도를 넣으면 정확도:", round(lag.score(future[cols], yt), 3))
same = (future["1시간전"] > 35).astype(int)         # "1시간 전과 같다"고 찍기
print("'1시간 전과 같다'고 찍으면:", round(accuracy_score(yt, same), 3))

97.7%는 놀라운 숫자입니다. 그런데 "1시간 전과 같다"고 찍기만 해도 97.9%입니다. 모델은 날씨에서 무엇을 배운 것이 아니라, 1시간 전 값을 거의 그대로 옮겨 적었습니다. 미세먼지는 한 시간 만에 크게 바뀌지 않기 때문입니다. 내일 아침의 미세먼지를 오늘 밤에 알리려는 앱이라면, 1시간 전 값은 아직 존재하지 않습니다. **점수가 너무 좋으면 축하하기 전에, 그 정보를 실제로 쓸 때도 알 수 있는지, 가장 단순한 찍기와 견주었는지 먼저 확인하십시오.** 13장 주제 C에서는 한 걸음 더 나아가, 오늘의 농도를 쓰되 "내일도 오늘과 같다"는 기준선과 겨룰 것입니다.

**5단계: 한계 쓰기.** 이 모델이 아직 모르는 것을 [표 9-4]처럼 세 가지 이상 쓰시오. 예를 들어 강의 자료의 날씨 파일에 넣지 않은 바람 방향(기상청 원자료에는 있음), 25곳 평균과 우리 동네의 차이, 실제 앱은 관측한 날씨가 아니라 예보를 써야 한다는 점을 생각해 보십시오.

**제출물.** 그림 2장(달별 "나쁨" 비율, 문턱값에 따른 놓침과 헛경보), 성적표 1개(기준선, 모델, 문턱값별 손해), 결론 5문장 이내.

**[표 9-5] 미니 프로젝트 채점 기준**

| 항목(비중) | 상 | 중 | 하 |
|---|---|---|---|
| 데이터 점검(20%) | 빈칸의 뜻, 24시 표기, 측정 위치, 등급 기준을 모두 밝힘 | 빈칸만 처리함 | 점검 없이 바로 모델을 만듦 |
| 기준선 비교(20%) | 시간 순서로 나누고 기준선과 견줌 | 기준선은 있으나 무작위로 나눔 | 기준선 없이 정확도만 씀 |
| 기법 적용(30%) | 모델, 검증 기간으로 고른 문턱값, 손해 계산이 맞음 | 모델은 맞으나 문턱값을 0.5로만 쓰거나 시험 기간으로 고름 | 코드가 돌지 않거나 결과가 틀림 |
| 해석과 한계(30%) | 97.7%를 왜 쓰면 안 되는지 설명하고 한계를 셋 이상 씀 | 한계를 한두 가지 씀 | 숫자만 나열함 |

**교수용 정답 예시.** 핵심 숫자는 이렇습니다. 합친 표 29,184시각, "나쁨" 11.4%, "매우 나쁨" 0.7%. 훈련(2023년) 8,676, 검증(2024년) 8,723, 시험(2025년~2026년 4월) 11,641시각. 시험 기간의 기준선(늘 괜찮음)은 86.4%이고, 문턱값 0.5의 로지스틱 회귀는 87.4%이지만 재현율이 14.5%뿐입니다. 놓침 × 3 + 헛경보로 손해를 정하면 검증 기간에서 문턱값 0.3이 골라지고, 시험 기간에는 재현율 38.3%, 손해 3,670(알림을 안 보내면 4,746)입니다. 1시간 전 농도를 넣으면 97.7%지만 "1시간 전과 같다"고 찍기만 해도 97.9%입니다. 결론의 예: "날씨만으로는 '나쁨'을 셋에 하나쯤 잡는다. 정확도는 기준선과 거의 같으니 재현율과 손해로 말해야 한다. 1시간 전 농도는 내일 아침 예측에는 쓸 수 없는 정보다. 25곳 평균이라 우리 동네와 다를 수 있고, 실제로는 예보 날씨를 써야 한다."

## ✏️ [연습문제 9-6] 정확도가 높은 모델이 더 좋은가

헛경보가 놓침보다 열 배 나쁘다고 할 때, 문턱값 0.5에서 숫자 개수 모델과 단어 로지스틱 회귀의 손해(헛경보 × 10 + 놓침)를 구하고 견주시오. 정확도와 견주어 결과를 설명하시오.

In [ ]:
# 여기에 직접 코드를 써 보십시오.

### 정답 코드

직접 해 본 뒤에 실행해 보십시오.

In [ ]:
th = 0.5
for name, pr in [("숫자 개수 하나", one.predict_proba(test[["숫자"]])[:, 1]),
                 ("로지스틱 회귀(단어)", prob)]:
    tn, fp, fn, tp = confusion_matrix(y, (pr >= th).astype(int)).ravel()
    print(f"{name}: 헛경보 {fp}  놓침 {fn}  손해 {10 * fp + fn}")